# Parte 2 — Consagração do Negócio: Entrega 1 — Modelagem Dimensional

Constrói o Star Schema com seis tabelas principais e três tabelas-ponte. As dimensões recebem chaves substitutas, e a fato reúne as métricas com uma linha por filme lançado. Ao final, são feitas checagens e uma consulta de teste da modelagem.


## Configuração da camada Gold

Usa o catálogo do projeto para definir os schemas Silver e Gold.


In [0]:
# Configura a origem Silver e o destino Gold da modelagem.

catalog = "cinedata_medallion"

bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

## Apoio — Funções de checagem da modelagem

Define as funções usadas para verificar unicidade e preenchimento de chaves. Os resultados ficam registrados com o nome da tabela e da regra.


In [0]:
# Estas funções serão usadas nas verificações das tabelas Gold.

from datetime import datetime
from pyspark.sql import Row, functions as F

dq_results = []

def dq_check(table_name, check_name, df, condition):
    total = df.count()
    failed = df.filter(~F.coalesce(condition, F.lit(False))).count()
    passed = failed == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=failed,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")
    return passed


def dq_check_unique(table_name, check_name, df, key_cols):
    total = df.count()
    duplicates = (
        df.groupBy(*key_cols)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    passed = duplicates == 0

    dq_results.append(
        Row(
            tabela=table_name,
            verificacao=check_name,
            total_linhas=total,
            quantidade_falhas=duplicates,
            aprovado=passed,
            verificado_em=datetime.now()
        )
    )

    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {duplicates} chaves duplicadas")
    return passed

## Entrega 1 — Leitura das tabelas Silver

Carrega os metadados, dados financeiros, métricas, gêneros, entidades e avaliações que serão usados na modelagem dimensional.


In [0]:
# Lê as tabelas tratadas que alimentam as dimensões, a fato e as pontes.

df_info = spark.table(f"{silver_schema}.tb_info_filmes")
df_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_generos = spark.table(f"{silver_schema}.tb_generos")
df_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")
df_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

## Entrega 1 — gold.dim_movies

Cria uma linha por filme com os metadados solicitados. id_filme é a chave natural em STRING; sk_movie_id é a chave substituta BIGINT. A tabela é gravada e relida para que os joins usem as chaves salvas.


In [0]:
# A chave substituta identifica o filme nos relacionamentos da Gold.

df_dim_movies = (
    df_info
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse")
    )
    .dropDuplicates(["id_filme"])
    .withColumn("sk_movie_id", F.monotonically_increasing_id())
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"
    )
)

df_dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

# Usa as chaves gravadas na tabela
df_dim_movies = spark.table(f"{gold_schema}.dim_movies")

display(df_dim_movies.limit(10))

sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
0,14564,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
1,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
2,50022,Super Troopers 2,2018-04-19,2018,100,en,Lançado,"When an international border dispute arises between the U.S. and Canada, the Super Troopers- Mac, Thorny, Foster, Rabbit and Farva, are called in to set up a new Highway Patrol station in the disputed area."
3,127380,FINDING DORY,2016-06-16,2016,97,en,Lançado,Dory is reunited with her friends Nemo and Marlin in the search for answers about her past. What can she remember? Who are her parents? And where did she learn to speak Whale?
4,137971,VH1 Divas 2009,2016-02-20,2016,0,en,Lançado,"VH1 Divas: An Honors Concert For The VH1 Save The Music Foundation aired live on September 17, 2009 at the Brooklyn Academy of Music (BAM) Howard Gilman Opera House in Brooklyn, New York. Paula Abdul hosted and performed. This concert, which featured a new and younger crop of divas as headliners, has not been released on CD or DVD."
5,139243,Moments choisis des Histoire(s) du cinéma,2023-05-11,2023,84,fr,Lançado,"A very personal look at the history of cinema directed, written and edited by Jean-Luc Godard in his Swiss residence in Rolle for ten years (1988-98); a monumental collage, constructed from film fragments, texts and quotations, photos and paintings, music and sound, and diverse readings; a critical, beautiful and melancholic vision of cinematographic art. (Abridged version of the original collection of eight short films)."
6,195590,Collide,2016-06-10,2016,98,en,Lançado,"An American backpacker gets involved with a ring of drug smugglers as their driver, and winds up on the run from his employers across Cologne’s autobahns."
7,228034,The Bronx Bull,2016-06-13,2016,94,en,Lançado,"\A combination """"\""""before the rage""""\"""" and """"\""""after the rage""""\"""" of world middleweight boxing champion Jake LaMotta's tumultuous life and times."""""""""
8,233863,Jimi Hendrix: Hear My Train a Comin',2020-09-11,2020,90,en,Lançado,"An account of the short life of genius musician Jimi Hendrix (1942-70), probably the most talented and influential guitarist of the twentieth century: his humble beginnings in Seattle, his time in New York, his rise to fame in swinging London… Live fast, love hard, die young."
9,242606,Keep Watching,2017-05-04,2017,90,en,Lançado,null


## Entrega 1 — gold.dim_genres

Cria o catálogo de gêneros distintos e gera sk_genre_id. A releitura da tabela permite usar as chaves gravadas na ponte de gêneros.


In [0]:
# Cada gênero distinto recebe sua chave substituta.

df_dim_genres = (
    df_generos
    .select(F.col("genero").alias("nome_genero"))
    .distinct()
    .withColumn("sk_genre_id", F.monotonically_increasing_id())
    .select("sk_genre_id", "nome_genero")
)

df_dim_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

df_dim_genres = spark.table(f"{gold_schema}.dim_genres")

display(df_dim_genres.limit(10))

sk_genre_id,nome_genero
0,Horror
1,Western
2,Comedy
3,Family
4,Romance
5,Science Fiction
6,Documentary
7,Fantasy
8,Music
9,Thriller


## Entrega 1 — gold.dim_people

Consolida Ator, Diretor e Roteirista. A combinação de nome e tipo define cada registro, permitindo que uma pessoa tenha mais de uma atuação. Gera sk_person_id e grava a dimensão.


In [0]:
# Deduplica nome e tipo de atuação antes de gerar a chave da pessoa.

df_dim_people = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    .withColumn("sk_person_id", F.monotonically_increasing_id())
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

df_dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

df_dim_people = spark.table(f"{gold_schema}.dim_people")

display(df_dim_people.limit(10))

sk_person_id,nome_pessoa,tipo_pessoa
0,David Fleeshman,Ator
1,Kimberly Quinn,Ator
2,Asuna Tomari,Ator
3,Darrin Rose,Ator
4,Pio Marmaï,Ator
5,Zach Callison,Ator
6,Mikhail Tikhonov,Ator
7,Hans Dagelet,Ator
8,José Ángel Bichir,Ator
9,Alexander Mørk Eidem,Ator


## Entrega 1 — gold.dim_companies

Seleciona as entidades do tipo Produtora, cria um catálogo de nomes distintos e gera sk_company_id.


In [0]:
# Cada produtora distinta recebe uma chave usada na ponte de empresas.

df_dim_companies = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.monotonically_increasing_id())
    .select("sk_company_id", "nome_produtora")
)

df_dim_companies.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

df_dim_companies = spark.table(f"{gold_schema}.dim_companies")

display(df_dim_companies.limit(10))

sk_company_id,nome_produtora
0,Ratpac Entertainment
1,Universal Pictures
2,Radio Silence
3,Endeavor Content
4,Ja Entertainment
5,Boy & Star
6,Box Fish Productions
7,Jiangsu Zhonglele Film And Television Media Co.
8,Mattel Creations
9,Cinedigm


## Entrega 1 — gold.dim_reviews

Resume as avaliações por filme com contagem e média arredondada em duas casas. A contagem inclui todas as avaliações; a média ignora notas NULL. Associa o resumo à chave do filme e gera sk_review_id.


In [0]:
# As avaliações individuais são agrupadas em um resumo por filme.

df_avaliacoes_resumo = (
    df_avaliacoes
    .withColumn("id_filme", F.col("id_filme").cast("string"))
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(
            F.avg("nota_usuario"), 2
        ).cast("double").alias("nota_media_usuarios")
    )
)

df_dim_reviews = (
    df_avaliacoes_resumo
    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )
    .withColumn("sk_review_id", F.monotonically_increasing_id())
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"
    )
)

df_dim_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

df_dim_reviews = spark.table(f"{gold_schema}.dim_reviews")

display(df_dim_reviews.limit(10))

sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
0,1,1,2.5
1,13,1,4.5
2,22,1,0.3
3,24,1,1.6
4,26,1,0.2
5,28,1,1.3
6,29,1,1.0
7,36,1,5.7
8,37,1,3.6
9,40,1,1.9


## Entrega 1 — gold.fact_movies_performance

Deduplica os dados financeiros e de engajamento por filme antes dos joins para preservar o grão. Usa filmes com status Lançado, exclui datas futuras e mantém data ausente quando o status informa lançamento. Os LEFT JOINs preservam filmes com métricas ausentes. Duplicatas com valores diferentes são reduzidas a uma versão sem critério de recência.


In [0]:
# A deduplicação antes dos joins mantém uma linha por filme na fato.

# Uma linha de dados financeiros por filme
df_financeiro_gold = (
    df_financeiro
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl")
    )
    .dropDuplicates(["id_filme"])
)

# Uma linha de dados de engajamento por filme
df_metricas_gold = (
    df_metricas
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("popularidade").cast("double").alias("popularidade"),
        F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        F.expr("try_cast(qtd_votos_tmdb AS INT)").alias("qtd_votos_tmdb"),
        F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        F.expr("try_cast(qtd_votos_imdb AS INT)").alias("qtd_votos_imdb")
    )
    .dropDuplicates(["id_filme"])
)

# Consolida somente os filmes com status Lançado
df_fact_movies_performance = (
    df_dim_movies
    .filter(
    (F.col("status_filme") == "Lançado")
    & (
        F.col("data_lancamento").isNull()
        | (F.col("data_lancamento") <= F.current_date())
    )
)
    .select("id_filme", "sk_movie_id")

    .join(df_financeiro_gold, on="id_filme", how="left")
    .join(df_metricas_gold, on="id_filme", how="left")

    .select(
        "sk_movie_id",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "orcamento_brl",
        "receita_brl",
        "lucro_brl",
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"
    )
)

df_fact_movies_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

display(df_fact_movies_performance.limit(10))

sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
0,25000000.00,83080890.00,58080890.00,124822500.00,414814575.68,289992075.68,24.584,4.966,2375,null,46286
1,null,null,null,null,null,null,2.98,7.1,null,7.8,212
2,null,18850674.00,null,null,94119530.21,null,12.698,5.854,417,null,31025
3,200000000.00,null,null,998580000.00,null,null,26.562,7.042,11429,7.2,339586
4,null,null,null,null,null,null,0.6,0.0,0,5.5,29
5,null,null,null,null,null,null,1.098,0.0,0,7.1,234
6,21500000.00,null,null,107347350.00,null,null,14.893,5.525,538,null,18311
7,7000000.00,null,null,34950300.00,null,null,4.4,5.1,13,null,816
8,null,null,null,null,null,null,2.797,8.1,13,7.6,604
9,null,null,null,null,null,null,null,5.071,261,null,null


## Entrega 1 — gold.bridge_movie_genre

Relaciona filmes e gêneros pelas chaves substitutas. Distinct garante um único par sk_movie_id e sk_genre_id para cada associação.


In [0]:
# A ponte representa os vários gêneros de um filme sem acrescentá-los à fato.

df_bridge_movie_genre = (
    df_generos
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )

    .join(
        df_dim_genres.withColumnRenamed("nome_genero", "genero"),
        on="genero",
        how="inner"
    )

    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

df_bridge_movie_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

display(df_bridge_movie_genre.limit(10))

sk_movie_id,sk_genre_id
9797,12
4933,17
34426,2
29482,3
24585,13
96120,10
96120,5
84875,5
69500,2
85740,7


## Entrega 1 — gold.bridge_movie_person

Relaciona filmes às pessoas conforme nome e tipo de atuação. Conserva pares únicos de sk_movie_id e sk_person_id.


In [0]:
# A ponte permite relacionar vários atores, diretores e roteiristas ao mesmo filme.

df_bridge_movie_person = (
    df_pessoas_empresas
    .filter(
        F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
    )
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )

    .join(
        df_dim_people
        .withColumnRenamed("nome_pessoa", "nome_entidade")
        .withColumnRenamed("tipo_pessoa", "tipo_entidade"),
        on=["nome_entidade", "tipo_entidade"],
        how="inner"
    )

    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

df_bridge_movie_person.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

display(df_bridge_movie_person.limit(10))

sk_movie_id,sk_person_id
18544,99306
95499,149125
12194,97
73205,99483
49233,149342
84776,149343
68745,99724
76614,476
15974,50454
61671,4


## Entrega 1 — gold.bridge_movie_company

Relaciona filmes às produtoras e conserva pares únicos de sk_movie_id e sk_company_id.


In [0]:
# A ponte registra as produtoras associadas a cada filme.

df_bridge_movie_company = (
    df_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .withColumn("id_filme", F.col("id_filme").cast("string"))

    .join(
        df_dim_movies.select("id_filme", "sk_movie_id"),
        on="id_filme",
        how="inner"
    )

    .join(
        df_dim_companies.withColumnRenamed(
            "nome_produtora", "nome_entidade"
        ),
        on="nome_entidade",
        how="inner"
    )

    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

df_bridge_movie_company.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

display(df_bridge_movie_company.limit(10))

sk_movie_id,sk_company_id
5671,4
96285,7019
14458,2785
9431,19745
81985,5622
49785,5624
51785,2801
71195,8481
82100,21120
24000,14117


## Validação da Entrega 1 — Chaves, grão e esquemas

Confere chaves e associações únicas, preenchimento das chaves, unicidade dos filmes e dos resumos de avaliações e tipos de pessoa permitidos. printSchema exibe os tipos gravados para conferência.


In [0]:
# Os testes verificam o grão das tabelas e as regras das chaves.

tabelas_e_chaves = [
    ("dim_movies", ["sk_movie_id"]),
    ("dim_genres", ["sk_genre_id"]),
    ("dim_people", ["sk_person_id"]),
    ("dim_companies", ["sk_company_id"]),
    ("dim_reviews", ["sk_review_id"]),
    ("fact_movies_performance", ["sk_movie_id"]),
    ("bridge_movie_genre", ["sk_movie_id", "sk_genre_id"]),
    ("bridge_movie_person", ["sk_movie_id", "sk_person_id"]),
    ("bridge_movie_company", ["sk_movie_id", "sk_company_id"])
]

for tabela, chaves in tabelas_e_chaves:
    df_tabela = spark.table(f"{gold_schema}.{tabela}")

    dq_check_unique(
        f"gold.{tabela}",
        "chave ou associação única",
        df_tabela,
        chaves
    )

    for chave in chaves:
        dq_check(
            f"gold.{tabela}",
            f"{chave} não nula",
            df_tabela,
            F.col(chave).isNotNull()
        )

    df_tabela.printSchema()

dq_check_unique(
    "gold.dim_movies",
    "um registro por id_filme",
    spark.table(f"{gold_schema}.dim_movies"),
    ["id_filme"]
)

dq_check_unique(
    "gold.dim_reviews",
    "um resumo de avaliações por filme",
    spark.table(f"{gold_schema}.dim_reviews"),
    ["sk_movie_id"]
)

dq_check(
    "gold.dim_people",
    "tipo de pessoa permitido",
    spark.table(f"{gold_schema}.dim_people"),
    F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista")
)

[PASS] gold.dim_movies | chave ou associação única | 0 chaves duplicadas
[PASS] gold.dim_movies | sk_movie_id não nula | 0/97879 linhas falharam
root
 |-- sk_movie_id: long (nullable = true)
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- ano_lancamento: integer (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)

[PASS] gold.dim_genres | chave ou associação única | 0 chaves duplicadas
[PASS] gold.dim_genres | sk_genre_id não nula | 0/19 linhas falharam
root
 |-- sk_genre_id: long (nullable = true)
 |-- nome_genero: string (nullable = true)

[PASS] gold.dim_people | chave ou associação única | 0 chaves duplicadas
[PASS] gold.dim_people | sk_person_id não nula | 0/420669 linhas falharam
root
 |-- sk_person_id: long (nullable = true)
 |-- nome_pessoa: string (nullable

True

## Validação da Entrega 1 — FKs e consulta do Star Schema

Verifica as oito relações de chave estrangeira com as dimensões. Depois, junta fato e dimensão de filmes e exibe receita, lucro e quantidade de filmes por idioma e ano, demonstrando o consumo do Star Schema.


In [0]:
# A consulta final testa o uso conjunto da fato e da dimensão de filmes.

# Verifica se as chaves estrangeiras existem nas dimensões
relacoes = [
    ("fact_movies_performance", "dim_movies", "sk_movie_id"),
    ("dim_reviews", "dim_movies", "sk_movie_id"),
    ("bridge_movie_genre", "dim_movies", "sk_movie_id"),
    ("bridge_movie_genre", "dim_genres", "sk_genre_id"),
    ("bridge_movie_person", "dim_movies", "sk_movie_id"),
    ("bridge_movie_person", "dim_people", "sk_person_id"),
    ("bridge_movie_company", "dim_movies", "sk_movie_id"),
    ("bridge_movie_company", "dim_companies", "sk_company_id")
]

for tabela, dimensao, chave in relacoes:
    df_origem = spark.table(f"{gold_schema}.{tabela}")

    df_chaves = (
        spark.table(f"{gold_schema}.{dimensao}")
        .select(chave)
        .withColumn("_fk_existe", F.lit(True))
    )

    df_validacao = df_origem.join(
        df_chaves,
        on=chave,
        how="left"
    )

    dq_check(
        f"gold.{tabela}",
        f"{chave} existe em {dimensao}",
        df_validacao,
        F.col("_fk_existe")
    )

# Consulta para testar o Star Schema, como no último print
df_resultado = (
    spark.table(
        f"{gold_schema}.fact_movies_performance"
    ).alias("f")

    .join(
        spark.table(f"{gold_schema}.dim_movies").alias("m"),
        on="sk_movie_id",
        how="inner"
    )

    .groupBy("m.idioma_original", "m.ano_lancamento")

    .agg(
        F.round(
            F.sum("f.receita_brl"), 2
        ).alias("receita_total_brl"),

        F.round(
            F.sum("f.lucro_brl"), 2
        ).alias("lucro_total_brl"),

        F.count("*").alias("qtd_filmes")
    )

    .orderBy(F.desc("receita_total_brl"))
)

display(df_resultado)

[PASS] gold.fact_movies_performance | sk_movie_id existe em dim_movies | 0/96460 linhas falharam
[PASS] gold.dim_reviews | sk_movie_id existe em dim_movies | 0/27303 linhas falharam
[PASS] gold.bridge_movie_genre | sk_movie_id existe em dim_movies | 0/140437 linhas falharam
[PASS] gold.bridge_movie_genre | sk_genre_id existe em dim_genres | 0/140437 linhas falharam
[PASS] gold.bridge_movie_person | sk_movie_id existe em dim_movies | 0/785020 linhas falharam
[PASS] gold.bridge_movie_person | sk_person_id existe em dim_people | 0/785020 linhas falharam
[PASS] gold.bridge_movie_company | sk_movie_id existe em dim_movies | 0/118925 linhas falharam
[PASS] gold.bridge_movie_company | sk_company_id existe em dim_companies | 0/118925 linhas falharam


idioma_original,ano_lancamento,receita_total_brl,lucro_total_brl,qtd_filmes
en,2017,128269995579.82,90659472213.69,7129
en,2016,113135222149.24,69650687105.96,6584
en,2019,109570644991.64,77945271467.14,6850
en,2018,106111561060.76,73520569728.33,6797
en,2022,80205675269.47,48740996633.68,5628
en,2023,56386458269.95,32856224615.55,4242
en,2021,49023933681.72,23107646581.34,5916
zh,2023,16978208455.44,3207701916.07,154
en,2020,16202301974.36,2963117354.99,6178
zh,2021,15625393329.62,7743573164.76,245
